In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score,log_loss
from sklearn.ensemble import GradientBoostingClassifier
import os
os.chdir('E:/Machine_Learning/Cases/Vehicle Silhouettes/')

In [6]:
vehicle = pd.read_csv('Vehicle.csv')
vehicle

,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra,Scat.Ra,Elong,Pr.Axis.Rect,Max.L.Rect,Sc.Var.Maxis,Sc.Var.maxis,Ra.Gyr,Skew.Maxis,Skew.maxis,Kurt.maxis,Kurt.Maxis,Holl.Ra,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841,93,39,87,183,64,8,169,40,20,134,200,422,149,72,7,25,188,195,saab
842,89,46,84,163,66,11,159,43,20,159,173,368,176,72,1,20,186,197,van
843,106,54,101,222,67,12,222,30,25,173,228,721,200,70,3,4,187,201,saab
844,86,36,78,146,58,7,135,50,18,124,155,270,148,66,0,25,190,195,saab


In [5]:
X,y = vehicle.drop('Class',axis=1) , vehicle['Class']
le = LabelEncoder()
le_y = le.fit_transform(y)
X_train,X_test,y_train,y_test = train_test_split(X,le_y,test_size=0.3,random_state=26,stratify=le_y)

In [7]:
gbm = GradientBoostingClassifier(random_state=26)
gbm.fit(X_train,y_train)
y_pred_prob = gbm.predict_proba(X_test)
print("AUC (OVO) : ",roc_auc_score(y_test,y_pred_prob,multi_class='ovo'))
print("AUC (OVR) : ",roc_auc_score(y_test,y_pred_prob,multi_class='ovr'))

AUC (OVO) :  0.9305661879520052
AUC (OVR) :  0.9294567746223928


In [8]:
gbm.predict(X_test)

array([2, 0, 0, 2, 0, 0, 0, 1, 2, 1, 3, 1, 1, 0, 1, 3, 2, 0, 0, 2, 2, 2,
       2, 1, 0, 3, 3, 0, 3, 1, 1, 1, 0, 0, 0, 0, 0, 3, 0, 1, 2, 1, 3, 3,
       2, 3, 2, 1, 0, 3, 3, 3, 1, 2, 0, 0, 1, 2, 0, 0, 1, 3, 1, 3, 3, 3,
       3, 2, 2, 1, 0, 1, 0, 0, 1, 1, 0, 0, 3, 3, 1, 3, 2, 3, 0, 3, 1, 3,
       3, 2, 2, 0, 2, 3, 3, 2, 3, 1, 1, 3, 2, 1, 0, 1, 1, 2, 0, 2, 1, 0,
       0, 2, 3, 1, 3, 2, 3, 1, 3, 2, 1, 3, 1, 2, 0, 1, 1, 1, 2, 0, 3, 2,
       2, 1, 2, 1, 3, 2, 3, 3, 3, 0, 3, 1, 2, 0, 1, 0, 2, 0, 2, 1, 0, 1,
       0, 3, 0, 2, 2, 1, 2, 2, 2, 3, 0, 2, 2, 1, 3, 1, 2, 0, 3, 3, 3, 3,
       0, 3, 2, 2, 0, 1, 0, 3, 0, 1, 0, 0, 3, 3, 3, 2, 3, 2, 0, 0, 1, 0,
       3, 2, 3, 0, 1, 1, 0, 3, 1, 0, 2, 0, 1, 2, 2, 3, 1, 1, 2, 0, 1, 2,
       3, 1, 1, 0, 3, 3, 2, 0, 0, 0, 2, 0, 3, 0, 0, 1, 2, 0, 2, 3, 3, 0,
       1, 3, 1, 1, 1, 2, 1, 2, 0, 0, 1, 3])

In [12]:
print("Log Loss: ",log_loss(y_test,y_pred_prob))

Log Loss:  0.5045919125374988


In [15]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = GradientBoostingClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            gbm.fit(X_train,y_train)
            y_pred_prob = gbm.predict_proba(X_test)
            scores.append([l,m,n,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','Log Loss'])
df_scores.sort_values(by='Log Loss',ascending=True)

100%|██████████| 10/10 [01:21<00:00,  8.20s/it]


,Learning_rate,Max_depth,n_estimators,Log Loss
63,0.504444,2.0,25,0.478272
52,0.405556,2.0,50,0.480414
17,0.108889,2.0,100,0.489373
51,0.405556,2.0,25,0.490844
40,0.306667,2.0,50,0.495539
...,...,...,...,...
109,0.900000,NaN,50,2.701495
108,0.900000,NaN,25,2.701495
86,0.702222,NaN,100,2.891586
85,0.702222,NaN,50,2.891586
